In [0]:
%sql
use catalog misgauravcatalog;
create database if not exists retaildb;
use retaildb; 

In [0]:
%sql
create table if not exists retaildb.orders_bronze
(
order_id int,
order_date string,
customer_id int,
order_status string,
filename string,
createdon timestamp
)
using delta
location "gs://databricksfolder/bronze/"
partitioned by (order_status)
TBLPROPERTIES(delta.enableChangeDataFeed = true);

In [0]:
%sql
create table if not exists retaildb.orders_silver
(
order_id int,
order_date date,
customer_id int,
order_status string,
order_year int GENERATED ALWAYS AS (YEAR(order_date)),
order_month int GENERATED ALWAYS AS (MONTH(order_date)),
createdon timestamp,
modifiedon timestamp
)
using delta
location "gs://databricksfolder/silver/"
partitioned by (order_status)
TBLPROPERTIES(delta.enableChangeDataFeed = true);

In [0]:
# 1. Create a 100% empty DataFrame with the exact schema matching your table
empty_silver_df = spark.sql("""
    SELECT 
      CAST(NULL AS INT) AS order_id,
      CAST(NULL AS DATE) AS order_date,
      CAST(NULL AS INT) AS customer_id,
      CAST(NULL AS STRING) AS order_status,
      CAST(NULL AS INT) AS order_year,
      CAST(NULL AS INT) AS order_month,
      CAST(NULL AS TIMESTAMP) AS createdon,
      CAST(NULL AS TIMESTAMP) AS modifiedon
    WHERE 1 = 0
""")

# 2. Write it directly to the GCS path. 
# This forces Delta Lake to physically build the _delta_log directory structure.
empty_silver_df.write \
    .format("delta") \
    .mode("append") \
    .save("gs://databricksfolder/silver/")

print("Physical storage log initialized successfully on GCS!")

In [0]:
%sql
create table if not exists retaildb.orders_gold
(
customer_id int,
order_status string,
order_year int,
num_orders int
)
using delta
location "gs://databricksfolder/gold/"
TBLPROPERTIES(delta.enableChangeDataFeed = true)

In [0]:
# 1. Create a 100% empty DataFrame with the exact schema matching your table
empty_gold_df = spark.sql("""
    SELECT 
      CAST(NULL AS INT) AS customer_id,
      CAST(NULL AS STRING) AS order_status,
      CAST(NULL AS INT) AS order_year,
      CAST(NULL AS INT) AS num_orders
    WHERE 1 = 0
""")

# 2. Write it directly to the GCS path. 
# This forces Delta Lake to physically build the _delta_log directory structure.
empty_gold_df.write \
    .format("delta") \
    .mode("append") \
    .save("gs://databricksfolder/gold/")

print("Physical storage log initialized successfully on GCS!")

In [0]:
%sql
copy into retaildb.orders_bronze from (
  select order_id::int,
  order_date::string,
  customer_id::int,
  order_status::string,
  _metadata.file_path as filename,
  current_timestamp() as createdon
  from 'gs://databricksfolder/raw/'
)
fileformat = CSV
format_options('header'='true');


In [0]:
%sql
describe history retaildb.orders_silver;

In [0]:
%sql
describe history retaildb.orders_gold;

In [0]:
# ====================================================================
# 1. Smart Source Routing: Read directly on Version 0, use CDF on Version 1+
# ====================================================================
history_df = spark.sql("DESCRIBE HISTORY retaildb.orders_bronze")
latest_version = history_df.select("version").first()[0]

if latest_version == 0:
    # Day 1: Read your 4 rows directly from the table since Version 0 has no CDF side-logs
    print("Processing initial baseline batch directly from Bronze data files...")
    changes_df = spark.read.table("retaildb.orders_bronze") \
        .filter("order_id > 0 AND customer_id > 0 AND order_status IN ('CLOSED', 'PENDING_PAYMENT')")
else:
    # Day 2+: Safely read incrementally using Change Data Feed starting from Version 1
    print(f"Processing incremental changes via CDF (Starting at Version 1, Current Table Version: {latest_version})...")
    changes_df = spark.read.format("delta") \
        .option("readChangeData", "true") \
        .option("startingVersion", 1) \
        .load("gs://databricksfolder/bronze/") \
        .filter("order_id > 0 AND customer_id > 0 AND order_status IN ('CLOSED', 'PENDING_PAYMENT')")

# Expose the dataframe back to your SQL engine
changes_df.createOrReplaceTempView("orders_bronze_changes")


# ====================================================================
# 2. Run Production MERGE
# ====================================================================
print("Executing Silver tier MERGE upsert logic...")
spark.sql("""
    MERGE INTO retaildb.orders_silver tgt
    USING orders_bronze_changes src ON tgt.order_id = src.order_id
    WHEN MATCHED THEN
      UPDATE SET tgt.order_status = src.order_status, tgt.customer_id = src.customer_id, tgt.modifiedon = CURRENT_TIMESTAMP()
    WHEN NOT MATCHED THEN
      INSERT (order_id, order_date, customer_id, order_status, createdon, modifiedon) 
      VALUES (src.order_id, src.order_date, src.customer_id, src.order_status, CURRENT_TIMESTAMP(), CURRENT_TIMESTAMP())
""")
print("Silver pipeline processed completely and cleanly!")

In [0]:
# ====================================================================
# 1. Smart Source Routing: Read directly on Versions 0 & 1, use CDF on Version 2+
# ====================================================================
history_df = spark.sql("DESCRIBE HISTORY retaildb.orders_silver")
latest_version = history_df.select("version").first()[0]

# Change condition: Versions 0 and 1 do not have row-level Change Data Feed side-logs
if latest_version <= 1:
    print(f"Processing initial baseline batches (Current Version: {latest_version}) directly from Silver data files...")
    changes_df = spark.sql("""
        SELECT customer_id, order_status, order_year, COUNT(order_id) AS num_orders
        FROM retaildb.orders_silver 
        GROUP BY customer_id, order_status, order_year
    """)
    changes_df.createOrReplaceTempView("orders_silver_changes")
    
else:
    print(f"Processing incremental changes via Silver CDF (Starting at Version 2, Current Version: {latest_version})...")
    # Day 3+: Safely read incrementally using Change Data Feed starting from Version 2
    cdf_df = spark.read.format("delta") \
        .option("readChangeData", "true") \
        .option("startingVersion", 2) \
        .load("gs://databricksfolder/silver/")
        
    cdf_df.createOrReplaceTempView("raw_silver_cdf")
    
    # Calculate net changes using SQL metadata tracking columns
    changes_df = spark.sql("""
        SELECT 
            customer_id, 
            order_status, 
            order_year,
            SUM(CASE 
                WHEN _change_type IN ('insert', 'update_postimage') THEN 1 
                WHEN _change_type IN ('delete', 'update_preimage') THEN -1 
                ELSE 0 
            END) AS net_new_orders
        FROM raw_silver_cdf
        GROUP BY customer_id, order_status, order_year
    """)
    changes_df.createOrReplaceTempView("orders_silver_changes")


# ====================================================================
# 2. Run Production MERGE (Handles Baselines vs Incremental dynamically)
# ====================================================================
print("Executing Gold tier MERGE upsert logic...")

if latest_version <= 1:
    # Direct match configuration for baseline loads
    spark.sql("""
        MERGE INTO retaildb.orders_gold tgt
        USING orders_silver_changes src 
        ON tgt.customer_id = src.customer_id AND tgt.order_status = src.order_status AND tgt.order_year = src.order_year
        WHEN MATCHED THEN
          UPDATE SET tgt.num_orders = src.num_orders
        WHEN NOT MATCHED THEN
          INSERT (customer_id, order_status, order_year, num_orders) VALUES (src.customer_id, src.order_status, src.order_year, src.num_orders)
    """)
else:
    # Incremental CDF balance adjustment math
    spark.sql("""
        MERGE INTO retaildb.orders_gold tgt
        USING orders_silver_changes src 
        ON tgt.customer_id = src.customer_id AND tgt.order_status = src.order_status AND tgt.order_year = src.order_year
        WHEN MATCHED THEN
          UPDATE SET tgt.num_orders = tgt.num_orders + src.net_new_orders
        WHEN NOT MATCHED THEN
          INSERT (customer_id, order_status, order_year, num_orders) VALUES (src.customer_id, src.order_status, src.order_year, src.net_new_orders)
    """)

print("Gold tier pipeline processed completely and cleanly!")